In [58]:
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from spotipy.oauth2 import SpotifyClientCredentials
import os
import base64
from requests import post, get
import json
import requests
from dotenv import load_dotenv

In [59]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [60]:
load_dotenv()
CLIENT_ID = os.getenv('CLIENT_ID')
CLIENT_SECRET = os.getenv('CLIENT_SECRET')

client_credentials_manager = SpotifyClientCredentials(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)
#REDIRECT_URI = "http://localhost:5000/callback"

In [61]:
def get_token():
    auth_string = CLIENT_ID + ':' + CLIENT_SECRET
    auth_bytes = auth_string.encode('utf-8')
    auth_base64 = str(base64.b64encode(auth_bytes), 'utf-8')

    url = 'https://accounts.spotify.com/api/token'
    headers = {'Authorization': 'Basic ' + auth_base64,
    'Content-Type' : 'application/x-www-form-urlencoded' }
    data = {'grant_type': 'client_credentials'}
    result = post(url, headers = headers, data=data)
    json_result = json.loads(result.content)
    token = json_result['access_token']

    return token

In [62]:
token = get_token()

In [6]:
my_track_ids_set = set()

In [9]:
greek_artists = [
    "Orfeas Peridis",
    "Melina Tanagri",
]

In [8]:
def get_tracks_by_genre(genre=None, limit=50, start_page=0, end_page=5, market=None):
    """
    genre: The genre you want to search for.
    limit: Number of items per page (max 50).
    start_page: The page index to start at.
    end_page: The page index to end at (non-inclusive).
    """
    track_ids = set()

    for page in range(start_page, end_page):
        offset = page * limit

        # Build base parameters
        search_params = {
            "type": "track",
            "limit": limit,
            "offset": offset
        }
        
        # Conditionally add market if provided
        if market is not None:
            search_params["market"] = market

        # same for the query:
        if genre is not None:
            search_params["q"] = f'genre:{genre}'
        else:
            search_params["q"] = f'market:{search_params["market"]}'
        
        # Pass the dictionary to sp.search
        results = sp.search(**search_params)

        items = results['tracks']['items']
        if not items: #if results have no items break early
            break

        for track in items:
            track_ids.add(track['id'])
        
        print(f"Fetched page {page}, got {len(items)} items.")

    return track_ids

In [10]:
total_artists_track_ids = []
for artist in greek_artists:
    results = sp.search(q=f"artist:{artist}", type="track", limit=50, market="GR")
    track_items = results['tracks']['items']
    total_artists_track_ids.append([track['id'] for track in track_items])
len(total_artists_track_ids)

2

In [369]:
#extracting the track_ids from above and adding them to the set
already_had = 0
for each_artist_tracks in total_artists_track_ids:
    for track_id in each_artist_tracks:
        if track_id in my_track_ids_set:
            already_had +=1 
        my_track_ids_set.add(track_id)
print(already_had)

125


In [193]:
# Getting track ids by genre(Greek):
for i in range(150, 155):
    my_track_ids = get_tracks_by_genre(genre=None, limit=4, start_page=i+10, end_page=i+11, market='GR')
    for track_id in my_track_ids:
        my_track_ids_set.add(track_id)
        print(track_id, track_id in my_track_ids)
    
    print(f"Number of track IDs: {len(my_track_ids)}")

Fetched page 160, got 4 items.
56nbr3ocdokoH1sTCItQxf True
44L1ioAGsEWhXClXwrUp06 True
7Crnt8QLU9cD7QyOAvyXHy True
00GlXgN81MaHGsLXlMeRsf True
Number of track IDs: 4
Fetched page 161, got 4 items.
2YueJMjDaReagzWKud2ri0 True
5NZDV8wWrPfggl2gE4Twz7 True
7iGcHo2l6nH25u8lDJUOsj True
6srMF9EQrNjf6het25EnAm True
Number of track IDs: 4
Fetched page 162, got 4 items.
22bJ1UO1PoYwETnKNDUWnd True
7B4xEbqaPmUZ6VLDvxPNnD True
6xeE7dGXefhcS1JgEVT6cG True
7bFSQGsuvX0vKB8MVpLUGf True
Number of track IDs: 4
Fetched page 163, got 4 items.
14St7deaerF9p3yRNEZ5nR True
6lokUQi0lVvZ5zywfe1Fw1 True
1IPEJpJZdflV8Zz2zUgDWf True
1pQbatDVRNyHCCwWj9qnhJ True
Number of track IDs: 4
Fetched page 164, got 4 items.
28QeQpwC6MF4zISH6TyjWx True
0dJlAMzCCVrEge0uDHMQrR True
0d4qR12eJhebbQaFEsUhPa True
3SjtnNowLqzpBAel7p9oG5 True
Number of track IDs: 4


In [41]:
track_ids = set()

for i in range (5):
    results = find_50_track_ids()
    for track in results['tracks']['items']:
        track_ids.add(track['id'])

In [42]:
len(track_ids)

50

In [370]:
#len(my_track_ids_set)
#30588 track ids from genre:Greek (and/or) market:GR (and/or) artist=*Greek artist name 
#aquired by myself(manual input) and with utilizing chatGPT o1 to fetch lists of greek singers/bands

30588

In [195]:
#"4ML8WdghEabh0eNU5P3w2W" in my_track_ids_set #(cross checking with Spotify API and my own set

True

In [372]:
# assign list (step finished, file created)
'''
l = list(my_track_ids_set)
total = 0
# open file
with open('song_ids.txt', 'w+') as f:
    
    # write elements of list
    for items in l:
        f.write('%s\n' %items)
        total += 1
    print("File written successfully")


# close the file
f.close()
print(total)
'''

File written successfully
30588


In [25]:
track_id_and_artist = []

In [52]:
####
#get the list of track IDs from the Json file
track_ids_list = []

In [53]:
json_data = [] # your list with json objects (dicts)

with open('track_ids.json') as json_file:
   json_data = json.load(json_file)

for item in json_data:
    track_ids_list.append(item[:-1])

In [23]:
quarter = len(track_ids_list)//4
quarter

7646

In [27]:
#use time.sleep to slow down the API calls
import time

In [28]:
for track_id in track_ids_list[:quarter]:
    results = sp.search(q='id:{track_id}')
    for track in results['tracks']['items']:
        track_id_and_artist.append([track['name'], track['artists'], track['id']])
    time.sleep(1)
    
track_id_and_artist

KeyboardInterrupt: 

In [ ]:
#above method with time.sleep(1) was used to avoid hitting spotify API rate limits (200 per minute)
#it is however really slow (7646 requests * 1 seconds => 2 hours 7 minutes)
#with method bellow, we split the track ids into batches of 50, and sleep after each batch
# [7646/50] = 153batches×1second/batch=153seconds ( 2.5 minutes).

In [34]:
batch_size = 50  # Spotify API allows a maximum of 50 track IDs per request
track_id_and_artist = []

for i in range(0, len(track_ids_list), batch_size):
    # Get the current batch of track IDs
    batch = track_ids_list[i:i + batch_size]

    try:
        # Join the batch IDs into a comma-separated string
        results = sp.tracks(",".join(batch))
        
        for track in results['tracks']:
            track_id_and_artist.append([track['name'], track['artists'], track['id']])
    except Exception as e:
        print(f"Error processing batch {i // batch_size}: {e}")
        continue  # Skip the batch on error and move to the next one

print("Finished processing all track IDs!")

Error processing batch 0: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 1: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 2: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 3: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 4: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 5: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 6: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 7: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 8: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 9: http status: 400, code:-1 - Unsupported URL / URI., reason: None
Error processing batch 10: http status: 400, code:-1 - Unsupported URL / URI., reason: Non

In [64]:
batch = track_ids_list[:50]  # Test with the first 50 IDs
batch_string = ",".join(batch)

try:
    results = sp.tracks(batch_string)
    print(results)  # Inspect the API response
except Exception as e:
    print(f"Error: {e}")

Error: http status: 400, code:-1 - Unsupported URL / URI., reason: None


In [63]:
url = f"https://api.spotify.com/v1/tracks?ids={batch_string}"
headers = {"Authorization": f"Bearer {token}"}
response = requests.get(url, headers=headers)
print(response.json())

{'tracks': [{'album': {'album_type': 'single', 'artists': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/1OJxI4NQfY6osMvhfRMMEX'}, 'href': 'https://api.spotify.com/v1/artists/1OJxI4NQfY6osMvhfRMMEX', 'id': '1OJxI4NQfY6osMvhfRMMEX', 'name': 'Dof Twogee', 'type': 'artist', 'uri': 'spotify:artist:1OJxI4NQfY6osMvhfRMMEX'}, {'external_urls': {'spotify': 'https://open.spotify.com/artist/5svarA8QyRUWetgH9ZouQq'}, 'href': 'https://api.spotify.com/v1/artists/5svarA8QyRUWetgH9ZouQq', 'id': '5svarA8QyRUWetgH9ZouQq', 'name': 'Sadam', 'type': 'artist', 'uri': 'spotify:artist:5svarA8QyRUWetgH9ZouQq'}], 'available_markets': ['AR', 'AU', 'AT', 'BE', 'BO', 'BR', 'BG', 'CA', 'CL', 'CO', 'CR', 'CY', 'CZ', 'DK', 'DO', 'DE', 'EC', 'EE', 'SV', 'FI', 'FR', 'GR', 'GT', 'HN', 'HK', 'HU', 'IS', 'IE', 'IT', 'LV', 'LT', 'LU', 'MY', 'MT', 'MX', 'NL', 'NZ', 'NI', 'NO', 'PA', 'PY', 'PE', 'PH', 'PL', 'PT', 'SG', 'SK', 'ES', 'SE', 'CH', 'TW', 'TR', 'UY', 'US', 'GB', 'AD', 'LI', 'MC', 'ID', 'JP', 'TH',

In [65]:
batch_string

'6wAtTDNQku5uE8ZmV1lpfv,0uOjIX6UULyr7FOvzBaeL6,0hR2DVULkbYErlbGCnBSiw,5cqsEaSklUJe117M2A7Feq,7HonfcsYMRIkEiPI4sTZMU,4T20C5S01rrXJLNTlOT3ou,5LkAdCASKpFWAZUONTuOiz,0CKgkvmuomIq5GneL5mCHf,72aws5vyKnPLqxi4nLveXc,1F63HZuMaZb90q2kgeM0G6,3ptcKvaWFthviHn2j8thSr,5Y3Mv8GGv3HpX7coCE0Pzd,67PR4447vyVw8dkYRCU7aP,1fVGRvKCutWn3ApIqSkjKJ,5tePNGQeEySWnOlJCHTO3y,5cnhmXk3sXAP0t51Qc8GOK,2UaohZdeDHSFjkc9qkTteB,4wapAg8jTJnTyeHf6Sunix,791WFhCnv7eZTPe5U8XbB4,7lirdOcXuV6nIPH1NTNwW8,1c6SuBSPzN6Oc82MRuJTgw,17UyNGoYXaGNT9xMrLJppB,4KIJOlcESIVl4RSBJGippF,2lDuBiX5e5Zad5LJpMHAdi,7HAVPwmS2f9eiuDO1nHMpT,6pkMxGsZiPkVJADgZipp6u,1tSUotPA3PIrU56eaHZ8NZ,1ywFB1yBGQZcLiItpKyF6Q,63elTzyIQMCtfFX19rk1N8,3f1jstQrVXrsuIEY595FrK,4wZepJJWGkPJ3hpgGMnL9r,7tRFd59sKEcAgNwT0GlxQz,1A2UR2GIQIFduMRFNDLjrE,4K8w2JVBrM36an1R7RMUhp,5ul9k4khYX11LMFKKrNrzO,6z6xXpsE4hwDEE4PIET7Rb,66v1pgbD2soUb3vTHZyF3p,6F1nHA51MsyaBBpnZzIsEb,65gv70Xo8afQcwk9KEt1Cu,0ZPd0LhcDGR8X1WrCo4nZW,2vZTsceOsMNQbFiVa9fZam,4X3SP6D8MQsgqFcrdlKddd,72nNJmpkk1CcKZOqCnEPG4,5uzZpr1dOZ

In [66]:
print(repr(batch_string))

'6wAtTDNQku5uE8ZmV1lpfv,0uOjIX6UULyr7FOvzBaeL6,0hR2DVULkbYErlbGCnBSiw,5cqsEaSklUJe117M2A7Feq,7HonfcsYMRIkEiPI4sTZMU,4T20C5S01rrXJLNTlOT3ou,5LkAdCASKpFWAZUONTuOiz,0CKgkvmuomIq5GneL5mCHf,72aws5vyKnPLqxi4nLveXc,1F63HZuMaZb90q2kgeM0G6,3ptcKvaWFthviHn2j8thSr,5Y3Mv8GGv3HpX7coCE0Pzd,67PR4447vyVw8dkYRCU7aP,1fVGRvKCutWn3ApIqSkjKJ,5tePNGQeEySWnOlJCHTO3y,5cnhmXk3sXAP0t51Qc8GOK,2UaohZdeDHSFjkc9qkTteB,4wapAg8jTJnTyeHf6Sunix,791WFhCnv7eZTPe5U8XbB4,7lirdOcXuV6nIPH1NTNwW8,1c6SuBSPzN6Oc82MRuJTgw,17UyNGoYXaGNT9xMrLJppB,4KIJOlcESIVl4RSBJGippF,2lDuBiX5e5Zad5LJpMHAdi,7HAVPwmS2f9eiuDO1nHMpT,6pkMxGsZiPkVJADgZipp6u,1tSUotPA3PIrU56eaHZ8NZ,1ywFB1yBGQZcLiItpKyF6Q,63elTzyIQMCtfFX19rk1N8,3f1jstQrVXrsuIEY595FrK,4wZepJJWGkPJ3hpgGMnL9r,7tRFd59sKEcAgNwT0GlxQz,1A2UR2GIQIFduMRFNDLjrE,4K8w2JVBrM36an1R7RMUhp,5ul9k4khYX11LMFKKrNrzO,6z6xXpsE4hwDEE4PIET7Rb,66v1pgbD2soUb3vTHZyF3p,6F1nHA51MsyaBBpnZzIsEb,65gv70Xo8afQcwk9KEt1Cu,0ZPd0LhcDGR8X1WrCo4nZW,2vZTsceOsMNQbFiVa9fZam,4X3SP6D8MQsgqFcrdlKddd,72nNJmpkk1CcKZOqCnEPG4,5uzZpr1dOZ

In [32]:
len(track_id_and_artist)//3

8080

In [67]:
import requests
# Batch processing with requests.get()
batch_size = 50
track_id_and_artist = []

for i in range(0, len(track_ids_list), batch_size):
    batch = track_ids_list[i:i + batch_size]
    batch_string = ",".join(batch)

    try:
        # Use direct requests.get() instead of Spotipy
        url = f"https://api.spotify.com/v1/tracks?ids={batch_string}"
        headers = {"Authorization": f"Bearer {token}"}
        response = requests.get(url, headers=headers)

        if response.status_code == 200:
            results = response.json()
            for track in results['tracks']:
                track_id_and_artist.append([track['name'], track['artists'], track['id']])
        else:
            print(f"Error: {response.status_code}, {response.text}")
    except Exception as e:
        print(f"Error processing batch {i // batch_size}: {e}")
        continue

print("Finished processing all track IDs!")


Finished processing all track IDs!


In [ ]:
##

In [69]:
len(track_id_and_artist)

30587

In [70]:
import json

# Save the current state of track_id_and_artist
with open("track_id_and_artist_backup.json", "w") as f:
    json.dump(track_id_and_artist, f)


In [31]:
audio_features = []

for track_id in my_track_ids_set:
    audio_features.append(get_audio_feature_by_track_id(token, track_id))

In [8]:
def get_auth_headers(token):
    return {'Authorization': 'Bearer ' + token}

In [ ]:
#token_info = client_credentials_manager.get_access_token(as_dict=True)
###
def get_audio_feature_by_track_id(token, track_id):
    url = f'https://api.spotify.com/v1/audio-features/{track_id}'
    headers = get_auth_headers(token)
    result = get(url, headers=headers)
    json_result = json.loads(result.content)
    return json_result
auth_token = token_info["access_token"]
####
headers = {"Authorization": f"Bearer {token}"}

# test a single known track
resp = requests.get("https://api.spotify.com/v1/tracks?/market=GR", headers=headers)
print("Status code:", resp.status_code)

In [1]:
#audio_features

In [35]:
#token == token_info["access_token"]

False